# 03 — Routing

Running real Dijkstra/A* over the graph, using `accessibility_weight` instead of raw `length`, to get an actual accessibility-aware route.

### Download the area and define `accessibility_weight` again

Same setup as notebook 02: tag whitelist expanded, Overpass mirror set, Financial District downloaded, and the weighting function from notes/02 redefined here so this notebook is self-contained.

In [ ]:
import osmnx as ox
import networkx as nx
import networkx as nx
import sys
sys.path.append("..")
from src.graph.tag_filters import find_edges_with_tag



ox.settings.cache_folder = "cache"
ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G = ox.graph_from_place("Financial District, Toronto, Canada", network_type="walk")

def accessibility_weight(attrs):
    base = attrs.get("length", 1)
    penalty = 0
    if attrs.get("wheelchair") == "no":
        penalty += 1000
    elif attrs.get("wheelchair") in ("yes", "designated"):
        penalty -= 2
    if attrs.get("surface") in ("gravel", "dirt", "grass", "ground"):
        penalty += 10
    return base + penalty

print(G.number_of_edges())

2372


### Store `acc_weight` as a real edge attribute

`nx.set_edge_attributes` writes a value onto every edge in one call, given a `{(u, v, k): value}` dict — the `k` is needed because this is a `MultiDiGraph` (more than one edge can connect the same two nodes), so `G.edges(keys=True, data=True)` is used instead of the plain `G.edges(data=True)` from before.

In [3]:
weights = {
    (u, v, k): accessibility_weight(data)
    for u, v, k, data in G.edges(keys=True, data=True)
}

nx.set_edge_attributes(G, weights, "acc_weight")


u, v, k, data = next(iter(G.edges(keys=True, data=True)))
print(data["length"], "->", data["acc_weight"])

10.053952309088126 -> 10.053952309088126


### Compare: shortest by raw length vs. shortest by accessibility weight

Picking two real nodes and running `nx.shortest_path` twice — once with `weight="length"` (ordinary shortest path) and once with `weight="acc_weight"` (accessibility-aware). If the two routes differ, it means the weighting actually changed the algorithm's decision, not just the numbers.

In [4]:
nodes = list(G.nodes())
source = nodes[0]
target = 13293811867
print("source:", source, " target:", target)

path_by_length = nx.shortest_path(G, source=source, target=target, weight="length")
cost_by_length = nx.shortest_path_length(G, source=source, target=target, weight="length")

path_by_acc = nx.shortest_path(G, source=source, target=target, weight="acc_weight")
cost_by_acc = nx.shortest_path_length(G, source=source, target=target, weight="acc_weight")

print("shortest by length:", len(path_by_length), "nodes, cost =", cost_by_length)
print("shortest by acc_weight:", len(path_by_acc), "nodes, cost =", cost_by_acc)
print("same route?", path_by_length == path_by_acc)

source: 29687486  target: 13293811867
shortest by length: 26 nodes, cost = 896.2170895282514
shortest by acc_weight: 26 nodes, cost = 896.2170895282514
same route? True


### Build a reusable helper: find edges by tag value

`find_edges_with_tag(G, tag, value)` (in `src/graph/tag_filters.py`) scans every edge and returns the `(u, v, k)` of every one matching a given tag/value — used here to find a real edge with `wheelchair: designated` so the source/target pair for the comparison above isn't picked blindly.

In [5]:
matches = find_edges_with_tag(G, "wheelchair", "designated")
print(len(matches))
print(matches[:5])

8
[(605209090, 13293811867, 0), (3376599895, 3376599898, 1), (3376599898, 3376599895, 1), (3570692004, 3570692005, 1), (3570692005, 3570692004, 1)]


### Validate the routing logic with a synthetic test case

Real-data testing above only shows the data exists and the arithmetic is right — it can't prove the routing *decision* changes, since the small `wheelchair: yes/designated` discount (-2) is too small to ever flip a choice on a ~900m real route. So: a small controlled graph where a `wheelchair: no` edge makes the physically shorter route the wrong one — to prove the algorithm actually avoids it, not just that the numbers compute correctly.

In [6]:
toy = nx.MultiDiGraph()

# Short route A -> B -> D, but not wheelchair accessible
toy.add_edge("A", "B", length=10, wheelchair="no")
toy.add_edge("B", "D", length=10, wheelchair="no")

# Long route A -> C -> D, fully accessible
toy.add_edge("A", "C", length=30, wheelchair="yes")
toy.add_edge("C", "D", length=30, wheelchair="yes")

toy_weights = {
    (u, v, k): accessibility_weight(data)
    for u, v, k, data in toy.edges(keys=True, data=True)
}
nx.set_edge_attributes(toy, toy_weights, "acc_weight")

route_by_length = nx.shortest_path(toy, "A", "D", weight="length")
route_by_acc = nx.shortest_path(toy, "A", "D", weight="acc_weight")

print("shortest by length:", route_by_length)
print("shortest by acc_weight:", route_by_acc)

shortest by length: ['A', 'B', 'D']
shortest by acc_weight: ['A', 'C', 'D']
